In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import silhouette_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM

import joblib

# ==========================
# LOAD DATASET
# ==========================

df = pd.read_csv("../data/diamonds.csv")

print(df.head())

# ==========================
# PREPROCESSING
# ==========================

if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

# Encoding
encoder_cut = LabelEncoder()
encoder_color = LabelEncoder()
encoder_clarity = LabelEncoder()

df["cut"] = encoder_cut.fit_transform(df["cut"])
df["color"] = encoder_color.fit_transform(df["color"])
df["clarity"] = encoder_clarity.fit_transform(df["clarity"])

# ==========================
# EDA
# ==========================

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True)
plt.show()

# ==========================
# SPLIT DATA
# ==========================

X = df.drop("price", axis=1)
y = df["price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================
# LINEAR REGRESSION
# ==========================

lr = LinearRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print("LINEAR REGRESSION")
print("MAE :", mean_absolute_error(y_test, pred_lr))
print("RMSE :", np.sqrt(mean_squared_error(y_test, pred_lr)))
print("R2 :", r2_score(y_test, pred_lr))

joblib.dump(lr, "../models/linear_regression_model.pkl")

# ==========================
# ANN
# ==========================

ann = Sequential()

ann.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
ann.add(Dense(32, activation='relu'))
ann.add(Dense(1))

ann.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

ann.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32
)

ann.save("../models/ann_model.h5")

# ==========================
# LSTM
# ==========================

X_train_lstm = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test_lstm = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

lstm = Sequential()

lstm.add(
    LSTM(
        50,
        activation='relu',
        input_shape=(X_train_lstm.shape[1],1)
    )
)

lstm.add(Dense(1))

lstm.compile(
    optimizer='adam',
    loss='mse'
)

lstm.fit(
    X_train_lstm,
    y_train,
    epochs=5,
    batch_size=32
)

lstm.save("../models/lstm_model.h5")

# ==========================
# KMEANS
# ==========================

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

cluster = kmeans.fit_predict(X_scaled)

df["Cluster"] = cluster

print("Silhouette Score :")
print(silhouette_score(X_scaled, cluster))

# ==========================
# BACKPROPAGATION
# ==========================

bp = Sequential()

bp.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
bp.add(Dense(64, activation='relu'))
bp.add(Dense(1))

bp.compile(
    optimizer='adam',
    loss='mse'
)

bp.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32
)

bp.save("../models/backpropagation_model.h5")

print("Semua model berhasil disimpan!")